In [6]:
# --- bootstrap: make `kg_tutorial` importable regardless of cwd ---
import sys
from pathlib import Path
_here = Path.cwd()
_root = _here if (_here / "kg_tutorial").exists() else _here.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
# -----------------------------------------------------------------


# Hour 0 — Setup & environment

> *15–20 minutes. Verify everything works, get your hands on the synthetic dataset, and meet the **Lotus case** that will follow you for the next 12 hours.*

If any cell below fails, fix the cause and re-run from the top — every subsequent hour assumes the checks in this hour pass.

**Reading companion:** [`docs/hour00.md`](../docs/hour00.md)


## What we verify

1. Python 3.11+ and the project is installed.
2. `ANTHROPIC_API_KEY` is set in `.env`.
3. The local embedding model loads and produces a vector.
4. Claude responds to a one-shot call.
5. The synthetic dataset exists (and we regenerate it if not).
6. Neo4j is running (optional — only needed from Hour 4 onwards).

Then we take a first look at the **Lotus case** — the corporate-KYC + UBO scenario the tutorial centres on.


## 1. Python and the project package

In [7]:
import sys
print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 11), "Need Python 3.11+"

import kg_tutorial
print("kg_tutorial:", kg_tutorial.__version__)


Python: 3.11.13
kg_tutorial: 0.1.0


## 2. Configuration and credentials

In [8]:
from kg_tutorial import config

config.verify()  # raises if ANTHROPIC_API_KEY is missing

print("Default model:    ", config.MODEL_DEFAULT)
print("Reasoning model:  ", config.MODEL_REASONING)
print("Embedding model:  ", config.EMBED_MODEL)
print("Data directory:   ", config.DATA_DIR)


Default model:     claude-sonnet-4-6
Reasoning model:   claude-opus-4-7
Embedding model:   BAAI/bge-small-en-v1.5
Data directory:    data/synthetic


## 3. Local embedding model

First call downloads ~134 MB to `~/.cache/huggingface/`. Subsequent calls are instant on Apple Silicon.


In [9]:
from kg_tutorial import embed

v = embed.embed("Ultimate Beneficial Owner")
print(f"Vector shape: {v.shape}, norm: {(v @ v) ** 0.5:.4f}")
print(f"Embedding dimension: {embed.dimension()}")


Vector shape: (384,), norm: 1.0000
Embedding dimension: 384


## 4. Claude is reachable

In [10]:
from kg_tutorial import llm

response = llm.ask(
    "In ten words or fewer, what is the FATF?",
    max_tokens=40,
)
print(response)


The FATF is an intergovernmental body combating money laundering and terrorist financing.


## 5. The synthetic dataset

If you've already run `uv run python -m kg_tutorial.data.generate`, this loads it. Otherwise the next cell generates it (deterministic — same seed every time).


In [11]:
from kg_tutorial.data import generate, load

try:
    bundle = load.load()
except FileNotFoundError:
    print("Generating dataset...")
    generate.write_dataset()
    load.load.cache_clear()
    bundle = load.load()

print("Dataset stats:")
for k, v in bundle.stats().items():
    print(f"  {k:>15}: {v}")


Generating dataset...
Dataset stats:
          persons: 20
         entities: 12
         controls: 6
         accounts: 1
         deposits: 31
        sanctions: 25
    adverse_media: 2
        documents: 7


## 6. Neo4j (optional now, required from Hour 4)

If Neo4j Desktop is running with the password from `.env.example`, the next cell will print a count of zero nodes. If you don't have Neo4j up yet, the cell will tell you — that's fine for Hours 0–3.

**Install:** download [Neo4j Desktop](https://neo4j.com/download/), create a project, add a *local DBMS* with password `neo4j-tutorial`, and click **Start**.


In [13]:
from kg_tutorial import config

try:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(
        config.NEO4J_URI,
        auth=(config.NEO4J_USER, config.NEO4J_PASSWORD),
    )
    with driver.session() as session:
        result = session.run("MATCH (n) RETURN count(n) AS n")
        n = result.single()["n"]
    driver.close()
    print(f"Neo4j is up. Current node count: {n}")
except Exception as e:
    print(f"Neo4j not reachable yet — that's OK for Hours 0–3.")
    print(f"  Cause: {type(e).__name__}: {e}")


Neo4j is up. Current node count: 0


## 7. Meet the Lotus case

This is the worked example for every hour. Memorize the shape of the chain — every subsequent lab refers back to it.


In [14]:
from kg_tutorial.data import load

b = load.load()

lotus_people = ["p_john_q_public", "p_alice_public", "p_maria_rossi", "p_tomas_velasco"]
lotus_entities = ["e_gamma_ops", "e_alphabeta_trading", "e_acme_holdings", "e_atlas_wirecorp"]

print("=== Persons ===")
for pid in lotus_people:
    p = load.person(pid)
    flag = " [PEP]" if p.is_pep else ""
    print(f"  {p.full_name}{flag} — {p.nationalities[0]}, resides {p.residence_country}")

print()
print("=== Entities ===")
for eid in lotus_entities:
    e = load.entity(eid)
    print(f"  {e.legal_name} ({e.entity_type.value}, {e.jurisdiction})")

print()
print("=== Control edges ===")
for c in b.controls:
    ctrl = c.controller_id
    ctrld = c.controlled_id
    pct = f" ({c.ownership_pct:.0f}%)" if c.ownership_pct is not None else ""
    print(f"  {ctrl}  --{c.control_type.value}{pct}-->  {ctrld}")


=== Persons ===
  John Q. Public [PEP] — GB, resides MC
  Alice Public, MP [PEP] — GB, resides GB
  Maria Rossi — IT, resides LI
  Tomás Velasco — PA, resides PA

=== Entities ===
  Gamma Operations GmbH (GmbH, LI)
  AlphaBeta Trading SA (SA, PA)
  ACME Holdings Ltd (SPV, KY)
  Atlas Wirecorp (Ltd, AE)

=== Control edges ===
  p_john_q_public  --UBO (50%)-->  e_acme_holdings
  e_acme_holdings  --Parent (100%)-->  e_alphabeta_trading
  e_alphabeta_trading  --Parent (75%)-->  e_gamma_ops
  p_maria_rossi  --Director-->  e_gamma_ops
  p_tomas_velasco  --Director-->  e_alphabeta_trading
  p_john_q_public  --Director-->  e_atlas_wirecorp


## 8. The question we will answer 13 ways

**Should this bank approve the account-opening application for Gamma Operations GmbH?**

Stop here and write down — in 2-3 lines — what you would want a control manager to know before signing off. By Hour 12 you'll have produced answers in:

- Gen 1 (Vector RAG) — and seen it fail.
- Gen 2 (Graph RAG) — and seen where it strains.
- Gen 3 (Hybrid + Agent) — with self-evaluation and full citation chain.

The same case, three architectures. The point of this tutorial is to make the difference *visceral*.


---

## Done

If every cell above ran without error, you're ready. Next: [Hour 1 — The retrieval problem](./hour01_concepts.ipynb).

If Neo4j wasn't up, leave it — install it before Hour 4.
